In [ ]:
# In[1]:

import json
from datetime import datetime
import pytz

# Load the task.json file
task_directory = 'rca100/cases/t039'
task_file_path = f'{task_directory}/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

# Extract alert context
alert_title = task_data.get('alert_title', None)
alert_window = task_data.get('alert_window', {})
alert_entity = task_data.get('alert_entity', {})
prompt_text = task_data.get('prompt_text', None)

# Convert alert window timestamps to datetime objects in UTC+8
timezone = pytz.timezone('Asia/Shanghai')
alert_start = datetime.fromisoformat(alert_window['start']).astimezone(timezone)
alert_end = datetime.fromisoformat(alert_window['end']).astimezone(timezone)

alert_title, alert_entity, alert_start, alert_end, prompt_text

```
Out[1]:
```
The alert is titled "checkout响应时间突增告警" (checkout response time spike alert). It occurred between 2026-04-28 09:18:30 and 2026-04-28 09:27:55 (UTC+8). The alert is related to the operation `checkout::/oteldemo.CheckoutService/PlaceOrder` in the `checkout` service, with a reported response time of 3355.24 ms. The user is requesting an analysis of the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

('checkout响应时间突增告警', {'entity_id': None, 'entity_name': None, 'entity_type': None, 'entity_domain': None}, datetime.datetime(2026, 4, 28, 9, 18, 30, 931868, tzinfo=<DstTzInfo 'Asia/Shanghai' CST+8:00:00 STD>), datetime.datetime(2026, 4, 28, 9, 27, 55, tzinfo=<DstTzInfo 'Asia/Shanghai' CST+8:00:00 STD>), '<code vibeops_object type="alert_event">\n<alert_event event_id="f7dbb0c94e4a368dc6cff604663a0c64" trans_id="dt80od6ml90vp4v592nmta72b2" rule_id="cb77edf9-9b7a-4b67-b29f-bfd88e358742" rule_name="checkout响应时间突增告警" entity_id="d219413245b68b297976412bbee076cf" entity_name="checkout::/oteldemo.CheckoutService/PlaceOrder" entity_type="apm.operation" entity_domain="apm" alert_time="2026-04-28T09:20:55+08:00" current_value="3355.241868578608" operation="/oteldemo.CheckoutService/PlaceOrder" service="checkout" service_id="<arms_svc_id>" workspace="rca-benchmark" region="cn-hongkong"/>\n</code> 帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# List all available KPIs and their associated components
kpi_components = metrics_data[['entity_set', 'entity_name', 'metric']].drop_duplicates()

kpi_components

```
Out[2]:
```
The `metrics.parquet` file contains various KPIs across multiple components. These include node-level metrics (e.g., `node_ready_status`, `node_memory_usage_rate`), service-level metrics (e.g., `error_rate` for specific service instances like `checkout` and `frontend`), and other operational metrics. There are a total of 323 unique combinations of components and KPIs. If you need a more detailed view of specific KPIs or components, let me know.

The original code execution output of IPython Kernel is also provided below for reference:

entity_set                      entity_name                  metric
0          k8s.node                                        node_ready_status
2002       k8s.node                                   node_memory_usage_rate
4147       k8s.node                                    node_not_ready_status
5720       k8s.node                                     node_disk_usage_rate
8008       k8s.node                                   node_pod_running_count
...             ...                              ...                     ...
91048  apm.instance        checkout-8559c59d4c-n5wwq              error_rate
91052  apm.instance        checkout-76dd8cf45b-657hm              error_rate
91056  apm.instance         frontend-86465b665-chd5b              error_rate
91079  apm.instance         frontend-86465b665-pjrnt              error_rate
91100  apm.instance  frontend-proxy-67d64dc4f8-4ptg6              error_rate

[323 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Ensure 'time' and 'value' columns are numeric for calculations
metrics_data['time'] = pd.to_numeric(metrics_data['time'], errors='coerce')
metrics_data['value'] = pd.to_numeric(metrics_data['value'], errors='coerce')

# Calculate the global P95 threshold for each 'component-KPI' combination
p95_thresholds = metrics_data.groupby(['entity_name', 'metric'])['value'].quantile(0.95).reset_index()
p95_thresholds.rename(columns={'value': 'p95_threshold'}, inplace=True)

p95_thresholds

```
Out[3]:
```
The global P95 thresholds for each 'component-KPI' combination have been calculated. This includes metrics like `apiserver_requests_per_second`, `avg_request_latency_seconds`, and `request_count` across various components such as `apiserver` and `shipping`. These thresholds will help identify anomalies by comparing actual metric values against these P95 thresholds. If you need details for specific components or KPIs, let me know.

The original code execution output of IPython Kernel is also provided below for reference:

entity_name                               metric  p95_threshold
0                                                     apiserver_concurrent_connections       7.000000
1                                                   apiserver_non_2xx_read_request_qps       0.160966
2                                                  apiserver_non_2xx_write_request_qps       0.422323
3                                                        apiserver_requests_per_second      28.499478
4                                                            cluster_failed_node_count       0.000000
..                                            ...                                  ...            ...
318   shipping::oteldemo.ShippingService/GetQuote          avg_request_latency_seconds       0.021987
319   shipping::oteldemo.ShippingService/GetQuote                        request_count   17452.600000
320   shipping::oteldemo.ShippingService/GetQuote                           slow_count     147.000000
321  shipping::oteldemo.ShippingService/ShipOrder          avg_request_latency_seconds       0.000011
322  shipping::oteldemo.ShippingService/ShipOrder                        request_count   17450.100000

[323 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Filter metrics data to include only the alert time window
alert_start_microseconds = int(alert_start.timestamp() * 1_000_000)
alert_end_microseconds = int(alert_end.timestamp() * 1_000_000)

filtered_metrics = metrics_data[
    (metrics_data['time'] >= alert_start_microseconds) &
    (metrics_data['time'] <= alert_end_microseconds)
]

# Merge filtered metrics with P95 thresholds to compare values
merged_metrics = filtered_metrics.merge(
    p95_thresholds,
    on=['entity_name', 'metric'],
    how='left'
)

# Identify anomalies where metric values exceed the P95 threshold
anomalies = merged_metrics[
    merged_metrics['value'] > merged_metrics['p95_threshold']
]

anomalies

```
Out[4]:
```
During the alert time window, 316 anomalies were identified where metric values exceeded their respective global P95 thresholds. These anomalies span across various components and metrics, including `node_disk_usage_rate` for Kubernetes nodes and `slow` or `mem` metrics for services like `frontend`, `inventory`, and `recommendation`. These anomalies indicate potential performance issues or resource constraints in these components. Further analysis is needed to pinpoint the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

time domain          entity_set entity_id     entity_name                metric         value                                     metric_set_id service  p95_threshold
936    1777339140000000    k8s            k8s.node                            node_disk_usage_rate  3.738298e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   3.736800e+01
937    1777339170000000    k8s            k8s.node                            node_disk_usage_rate  3.738312e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   3.736800e+01
938    1777339200000000    k8s            k8s.node                            node_disk_usage_rate  3.738990e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   3.736800e+01
939    1777339230000000    k8s            k8s.node                            node_disk_usage_rate  3.739005e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   3.736800e+01
940    1777339260000000    k8s            k8s.node                            node_disk_usage_rate  3.739016e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   3.736800e+01
...                 ...    ...                 ...       ...             ...                   ...           ...                                               ...     ...            ...
10738  1777339650000000    apm  apm.service.legacy                  frontend                  slow  5.900000e+02                                              None    None   5.833000e+02
10745  1777339650000000    apm  apm.service.legacy                 inventory                   mem  6.296672e+08                                              None    None   6.296532e+08
10748  1777339650000000    apm  apm.service.legacy            load-generator                   mem  1.906217e+09                                              None    None   1.906213e+09
10759  1777339650000000    apm  apm.service.legacy                     quote                   mem  6.104405e+07                                              None    None   6.103736e+07
10763  1777339650000000    apm  apm.service.legacy            recommendation                   mem  1.962749e+08                                              None    None   1.962675e+08

[316 rows x 10 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = f'{task_directory}/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Ensure 'startTime', 'endTime', and 'duration' columns are numeric
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')
traces_data['duration'] = pd.to_numeric(traces_data['duration'], errors='coerce')

# Filter traces for the alert time window
alert_start_nanoseconds = alert_start_microseconds * 1_000
alert_end_nanoseconds = alert_end_microseconds * 1_000

filtered_traces = traces_data[
    (traces_data['startTime'] >= alert_start_nanoseconds) &
    (traces_data['endTime'] <= alert_end_nanoseconds)
]

# Filter traces related to the `checkout` service and its operation `/oteldemo.CheckoutService/PlaceOrder`
checkout_traces = filtered_traces[
    (filtered_traces['serviceName'] == 'checkout') &
    (filtered_traces['spanName'] == '/oteldemo.CheckoutService/PlaceOrder')
]

# Look for errors or high latencies in the filtered traces
checkout_traces_with_issues = checkout_traces[
    (checkout_traces['statusCode'] != 0) | (checkout_traces['duration'] > checkout_traces['duration'].quantile(0.95))
]

checkout_traces_with_issues

```
Out[5]:
```
No traces related to the `checkout` service and its operation `/oteldemo.CheckoutService/PlaceOrder` during the alert time window showed errors or high latencies. This suggests that the `checkout` service itself may not be the most downstream faulty component in the call chain. Further investigation into upstream or related components is needed to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, hostname, statusCode, statusMessage, traceState, resources, attributes, __topic__, __tag__:__pack_id__, events, ip, links]
Index: []

[0 rows x 21 columns]```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file_path = f'{task_directory}/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Ensure '_time_' column is in numeric format
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs for the alert time window
filtered_logs = logs_data[
    (logs_data['_time_'] >= alert_start_microseconds) &
    (logs_data['_time_'] <= alert_end_microseconds)
]

# Search for error or warning logs related to the `checkout`, `frontend`, `inventory`, and `recommendation` services
services_to_check = ['checkout', 'frontend', 'inventory', 'recommendation']
error_warning_logs = filtered_logs[
    (filtered_logs['_container_name_'].str.contains('|'.join(services_to_check), case=False, na=False)) &
    (filtered_logs['content'].str.contains('error|warning', case=False, na=False))
]

error_warning_logs

```
Out[6]:
```
No error or warning logs were found for the `checkout`, `frontend`, `inventory`, or `recommendation` services during the alert time window. This indicates that no critical issues or patterns were logged by these services during the incident. Further investigation into other telemetry data or components may be required to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _namespace_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```

In [ ]:
# In[7]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[7]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[8]:

# Inspect the columns of the events DataFrame to identify the correct timestamp column
events_data_columns = events_data.columns

events_data_columns

```
Out[8]:
```
The `events.parquet` file contains columns such as `eventId`, `hostname`, `level`, `pod_id`, `pod_name`, `clusterId`, and `clusterName`. However, it does not include a direct timestamp column for filtering by the alert time window. Further investigation is needed to determine how to filter events based on time.

The original code execution output of IPython Kernel is also provided below for reference:

Index(['eventId', 'hostname', 'level', 'pod_id', 'pod_name', 'clusterId', 'clusterName', '__topic__'], dtype='object')```
```

In [ ]:
# In[9]:

import json

# Parse the `eventId` column to extract relevant fields, including timestamps
events_data['parsed_event'] = events_data['eventId'].apply(json.loads)
events_data['event_time'] = events_data['parsed_event'].apply(lambda x: x.get('metadata', {}).get('creationTimestamp'))
events_data['reason'] = events_data['parsed_event'].apply(lambda x: x.get('reason'))
events_data['message'] = events_data['parsed_event'].apply(lambda x: x.get('message'))

# Convert `event_time` to datetime and filter for the alert time window
events_data['event_time'] = pd.to_datetime(events_data['event_time'], errors='coerce').dt.tz_convert('Asia/Shanghai')
filtered_events = events_data[
    (events_data['event_time'] >= alert_start) &
    (events_data['event_time'] <= alert_end)
]

# Focus on node-level issues, pod scheduling problems, or other anomalies
relevant_events = filtered_events[
    filtered_events['reason'].str.contains('Node|Pod|Unschedulable|Failed', case=False, na=False)
]

relevant_events[['event_time', 'reason', 'message', 'hostname', 'pod_name']]

```
Out[9]:
```
No relevant events related to node-level issues, pod scheduling problems, or other anomalies were found in the `events.parquet` file during the alert time window. This suggests that there were no significant Kubernetes-level issues contributing to the identified metrics anomalies. Further investigation into other telemetry data may be necessary to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [event_time, reason, message, hostname, pod_name]
Index: []```
```